# GPT-2 from Scratch with Keras + JAX

## Introduction

In this notebook, we'll build a GPT-2 style transformer model from scratch using **Keras** with **JAX** backend. This is a character-level language model that learns to generate text.

### What We'll Learn

- How **token embeddings** and **position embeddings** work
- Building **self-attention mechanisms** from scratch
- Implementing **multi-head attention**
- Creating **transformer blocks** with residual connections
- Training with **JAX** for high performance
- Generating text with the trained model

### Why Keras + JAX?

- **JAX**: Fast automatic differentiation and XLA compilation
- **Keras 3**: Clean API that works with JAX backend
- **Educational**: Shows how transformers work at a low level

## Configuration

We define all hyperparameters in one place for easy experimentation.

In [ ]:
CONFIG = {
    # Model architecture
    "seq_len": 128,  # Context window size (sequence length)
    "embed_size": 64,  # Embedding dimension
    "num_heads": 2,  # Number of attention heads (must divide embed_size)
    "num_layers": 4,  # Number of transformer blocks
    "ff_dim": 256,  # Feed-forward network dimension (typically 4x embed_size)
    
    # Training
    "batch_size": 64,  # Samples per training batch
    "learning_rate": 1e-3,  # Learning rate for AdamW
    "max_steps": 2000,  # Number of training steps
    
    # Regularization
    "layer_norm_enabled": True,  # Use layer normalization
    
    # Reproducibility
    "seed": 42,  # Random seed
}

## Setup JAX and Keras

Configure Keras to use JAX backend and set random seeds for reproducibility.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'

import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import keras

# Set random seeds
np.random.seed(CONFIG['seed'])
key = random.PRNGKey(CONFIG['seed'])

print(f"JAX backend: {jax.default_backend()}")
print(f"Keras backend: {keras.backend.backend()}")
print(f"Devices: {jax.devices()}")

## Load Dataset

We'll use the Lord of the Rings text dataset for training our language model.

In [ ]:
# Load dataset
with open("data/lotr.txt", "r", encoding="latin-1") as f:
    DATASET_TEXT = f.read()

print(f"Dataset size: {len(DATASET_TEXT):,} characters")
print(f"\nFirst 200 characters:\n{DATASET_TEXT[:200]}")

## Create Character Tokenizer

We'll build a simple character-level tokenizer that maps each unique character to an integer.

In [ ]:
# Get unique characters and create mappings
chars = sorted(list(set(DATASET_TEXT)))
vocab_size = len(chars)
CONFIG["vocab_size"] = vocab_size

# Create encode/decode functions
ctoi = {c: i for i, c in enumerate(chars)}
itoc = {i: c for i, c in enumerate(chars)}
encode = lambda text: [ctoi[c] for c in text]
decode = lambda tokens: "".join([itoc[i] for i in tokens])

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {chars}")
print(f"\nTest encoding/decoding:")
test_text = "hello world"
encoded = encode(test_text)
print(f"'{test_text}' -> {encoded} -> '{decode(encoded)}'")

## Token Embeddings

Each token (character) is mapped to a dense vector representation using an **embedding table**.

In [ ]:
from keras import layers

# Create token embedding layer
token_embedding = layers.Embedding(
    input_dim=vocab_size,
    output_dim=CONFIG["embed_size"],
    name="token_embedding"
)

# Test with a sample
sample_text = "hello"
sample_tokens = jnp.array(encode(sample_text))
sample_embeddings = token_embedding(sample_tokens)

print(f"Input: '{sample_text}'")
print(f"Tokens shape: {sample_tokens.shape}")
print(f"Embeddings shape: {sample_embeddings.shape}")
print(f"Each character is now a {CONFIG['embed_size']}-dimensional vector")

## Position Embeddings

Since transformers have no inherent notion of sequence order, we add **positional information** using learned position embeddings.

In [ ]:
# Create position embedding layer
position_embedding = layers.Embedding(
    input_dim=CONFIG["seq_len"],
    output_dim=CONFIG["embed_size"],
    name="position_embedding"
)

# Test position embeddings
positions = jnp.arange(len(sample_tokens))
pos_embeddings = position_embedding(positions)

print(f"Positions: {positions}")
print(f"Position embeddings shape: {pos_embeddings.shape}")

## Combined Embeddings

We **add** token embeddings and position embeddings to get the final input representation.

In [ ]:
# Combine token and position embeddings
combined_embeddings = sample_embeddings + pos_embeddings

print(f"Token embeddings shape: {sample_embeddings.shape}")
print(f"Position embeddings shape: {pos_embeddings.shape}")
print(f"Combined embeddings shape: {combined_embeddings.shape}")
print(f"\nThis combined representation contains both 'what' (token) and 'where' (position) information")

## Prepare Dataset for Training

Create input-output pairs for training: each input sequence predicts the next character at each position.

In [ ]:
# Tokenize entire dataset
DATASET_TOKENS = encode(DATASET_TEXT)

# Create dataset of overlapping sequences
def create_sequences(tokens, seq_len):
    """Create input-output pairs for language modeling."""
    xs, ys = [], []
    for i in range(len(tokens) - seq_len):
        x = tokens[i:i+seq_len]
        y = tokens[i+1:i+seq_len+1]  # Shifted by 1
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

X, Y = create_sequences(DATASET_TOKENS, CONFIG["seq_len"])

print(f"Number of training sequences: {len(X):,}")
print(f"Input shape: {X.shape}")
print(f"Output shape: {Y.shape}")
print(f"\nExample:")
print(f"Input:  '{decode(X[0].tolist())[:50]}...'")
print(f"Target: '{decode(Y[0].tolist())[:50]}...'")

## Batch Sampling

For efficient training, we'll randomly sample batches from our dataset.

In [ ]:
def sample_batch(X, Y, batch_size, rng_key):
    """Sample a random batch from the dataset."""
    n_samples = len(X)
    indices = random.choice(rng_key, n_samples, shape=(batch_size,), replace=False)
    return X[indices], Y[indices]

# Test batch sampling
key, subkey = random.split(key)
batch_x, batch_y = sample_batch(X, Y, batch_size=4, rng_key=subkey)

print(f"Batch input shape: {batch_x.shape}")
print(f"Batch output shape: {batch_y.shape}")
print(f"\nFirst sequence in batch:")
print(f"Input:  '{decode(batch_x[0].tolist())[:80]}...'")
print(f"Target: '{decode(batch_y[0].tolist())[:80]}...'")

## Causal Self-Attention

The core of the transformer is **self-attention**. For language modeling, we use **causal** (masked) attention so each position can only attend to previous positions.

In [ ]:
import keras.ops as ops

class CausalSelfAttention(layers.Layer):
    """Single-head causal self-attention."""
    
    def __init__(self, head_size, **kwargs):
        super().__init__(**kwargs)
        self.head_size = head_size
        
    def build(self, input_shape):
        embed_size = input_shape[-1]
        
        # Combined Q, K, V projection
        self.qkv_proj = layers.Dense(self.head_size * 3, use_bias=False)
        
        # Create causal mask (lower triangular)
        self.seq_len = CONFIG["seq_len"]
        
    def call(self, x):
        B, T, C = ops.shape(x)[0], ops.shape(x)[1], ops.shape(x)[2]
        
        # Compute Q, K, V
        qkv = self.qkv_proj(x)  # (B, T, 3*head_size)
        q, k, v = ops.split(qkv, 3, axis=-1)
        
        # Compute attention scores
        scores = ops.matmul(q, ops.transpose(k, axes=[0, 2, 1]))  # (B, T, T)
        scores = scores / ops.sqrt(ops.cast(self.head_size, dtype=scores.dtype))
        
        # Apply causal mask
        mask = ops.tril(ops.ones((T, T)))
        mask = ops.cast(mask, dtype=scores.dtype)
        scores = ops.where(mask == 0, -1e9, scores)
        
        # Apply softmax to get attention weights
        attn_weights = ops.softmax(scores, axis=-1)
        
        # Apply attention to values
        out = ops.matmul(attn_weights, v)  # (B, T, head_size)
        
        return out

# Test attention layer
attn_layer = CausalSelfAttention(head_size=32)
test_input = jnp.ones((2, 10, 64))  # (batch=2, seq_len=10, embed_size=64)
attn_output = attn_layer(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {attn_output.shape}")

## Multi-Head Attention

Instead of a single attention head, we use **multiple heads** that learn different attention patterns, then concatenate and project their outputs.

In [ ]:
class MultiHeadAttention(layers.Layer):
    """Multi-head causal self-attention."""
    
    def __init__(self, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        
    def build(self, input_shape):
        embed_size = input_shape[-1]
        assert embed_size % self.num_heads == 0, "embed_size must be divisible by num_heads"
        
        self.head_size = embed_size // self.num_heads
        self.embed_size = embed_size
        
        # Combined Q, K, V projection for all heads
        self.qkv_proj = layers.Dense(embed_size * 3, use_bias=False)
        
        # Output projection
        self.out_proj = layers.Dense(embed_size, use_bias=False)
        
    def call(self, x):
        B = ops.shape(x)[0]
        T = ops.shape(x)[1]
        C = self.embed_size
        
        # Compute Q, K, V for all heads at once
        qkv = self.qkv_proj(x)  # (B, T, 3*C)
        qkv = ops.reshape(qkv, (B, T, 3, self.num_heads, self.head_size))
        qkv = ops.transpose(qkv, axes=[2, 0, 3, 1, 4])  # (3, B, num_heads, T, head_size)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Compute attention scores
        scores = ops.matmul(q, ops.transpose(k, axes=[0, 1, 3, 2]))  # (B, num_heads, T, T)
        scores = scores / ops.sqrt(ops.cast(self.head_size, dtype=scores.dtype))
        
        # Apply causal mask
        mask = ops.tril(ops.ones((T, T)))
        mask = ops.cast(mask, dtype=scores.dtype)
        scores = ops.where(mask == 0, -1e9, scores)
        
        # Apply softmax
        attn_weights = ops.softmax(scores, axis=-1)
        
        # Apply attention to values
        out = ops.matmul(attn_weights, v)  # (B, num_heads, T, head_size)
        
        # Concatenate heads
        out = ops.transpose(out, axes=[0, 2, 1, 3])  # (B, T, num_heads, head_size)
        out = ops.reshape(out, (B, T, C))  # (B, T, C)
        
        # Output projection
        out = self.out_proj(out)
        
        return out

# Test multi-head attention
mha_layer = MultiHeadAttention(num_heads=CONFIG["num_heads"])
test_input = jnp.ones((2, 10, CONFIG["embed_size"]))
mha_output = mha_layer(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {mha_output.shape}")
print(f"Number of heads: {CONFIG['num_heads']}")

## Feed-Forward Network

After attention, each position goes through a **feed-forward network** that processes information independently at each position.

In [ ]:
class FeedForward(layers.Layer):
    """Position-wise feed-forward network."""
    
    def __init__(self, ff_dim, **kwargs):
        super().__init__(**kwargs)
        self.ff_dim = ff_dim
        
    def build(self, input_shape):
        embed_size = input_shape[-1]
        self.fc1 = layers.Dense(self.ff_dim)
        self.fc2 = layers.Dense(embed_size)
        
    def call(self, x):
        x = self.fc1(x)
        x = ops.gelu(x)
        x = self.fc2(x)
        return x

# Test feed-forward network
ffn_layer = FeedForward(ff_dim=CONFIG["ff_dim"])
test_input = jnp.ones((2, 10, CONFIG["embed_size"]))
ffn_output = ffn_layer(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {ffn_output.shape}")
print(f"Feed-forward expands to {CONFIG['ff_dim']} dimensions internally")

## Transformer Block

A transformer block combines **multi-head attention** and **feed-forward network** with **residual connections** and **layer normalization**.

In [ ]:
class TransformerBlock(layers.Layer):
    """A single transformer block with attention and feed-forward."""
    
    def __init__(self, num_heads, ff_dim, layer_norm_enabled=True, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.layer_norm_enabled = layer_norm_enabled
        
    def build(self, input_shape):
        self.attn = MultiHeadAttention(self.num_heads)
        self.ffn = FeedForward(self.ff_dim)
        
        if self.layer_norm_enabled:
            self.ln1 = layers.LayerNormalization(epsilon=1e-5)
            self.ln2 = layers.LayerNormalization(epsilon=1e-5)
        
    def call(self, x):
        # Multi-head attention with residual connection
        if self.layer_norm_enabled:
            x = x + self.attn(self.ln1(x))
        else:
            x = x + self.attn(x)
        
        # Feed-forward with residual connection
        if self.layer_norm_enabled:
            x = x + self.ffn(self.ln2(x))
        else:
            x = x + self.ffn(x)
        
        return x

# Test transformer block
block = TransformerBlock(
    num_heads=CONFIG["num_heads"],
    ff_dim=CONFIG["ff_dim"],
    layer_norm_enabled=CONFIG["layer_norm_enabled"]
)
test_input = jnp.ones((2, 10, CONFIG["embed_size"]))
block_output = block(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {block_output.shape}")

## GPT-2 Model

Now we combine everything into the full **GPT-2 model**: embeddings → transformer blocks → language modeling head.

In [ ]:
class GPT2Model(keras.Model):
    """GPT-2 style transformer for language modeling."""
    
    def __init__(self, config, **kwargs):
        super().__init__(**kwargs)
        self.config = config
        
        # Embedding layers
        self.token_embedding = layers.Embedding(
            input_dim=config["vocab_size"],
            output_dim=config["embed_size"]
        )
        self.position_embedding = layers.Embedding(
            input_dim=config["seq_len"],
            output_dim=config["embed_size"]
        )
        
        # Transformer blocks
        self.blocks = [
            TransformerBlock(
                num_heads=config["num_heads"],
                ff_dim=config["ff_dim"],
                layer_norm_enabled=config["layer_norm_enabled"]
            )
            for _ in range(config["num_layers"])
        ]
        
        # Final layer norm
        if config["layer_norm_enabled"]:
            self.ln_final = layers.LayerNormalization(epsilon=1e-5)
        
        # Language modeling head
        self.lm_head = layers.Dense(config["vocab_size"])
        
    def call(self, x):
        B, T = ops.shape(x)[0], ops.shape(x)[1]
        
        # Get embeddings
        tok_emb = self.token_embedding(x)  # (B, T, embed_size)
        pos = ops.arange(T)
        pos_emb = self.position_embedding(pos)  # (T, embed_size)
        
        # Combine embeddings
        x = tok_emb + pos_emb
        
        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Final layer norm
        if self.config["layer_norm_enabled"]:
            x = self.ln_final(x)
        
        # Project to vocabulary
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        return logits

# Create model
model = GPT2Model(CONFIG)

# Test forward pass
test_input = jnp.array([[1, 2, 3, 4, 5]])  # (batch=1, seq_len=5)
test_output = model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"\nModel predicts next-token probabilities for each position")

## Model Summary

Let's inspect the model architecture and parameter count.

In [ ]:
# Build model with correct input shape
model.build((None, CONFIG["seq_len"]))
model.summary()

# Count parameters
total_params = sum([np.prod(p.shape) for p in model.trainable_weights])
print(f"\nTotal trainable parameters: {total_params:,}")

## Training with JAX

We'll implement a custom training loop using JAX for maximum performance and control.

In [ ]:
import optax
from tqdm import tqdm

# Initialize optimizer
optimizer = optax.adamw(learning_rate=CONFIG["learning_rate"])

# Get model parameters
params = [v.value for v in model.trainable_weights]
opt_state = optimizer.init(params)

# Loss function
def compute_loss(params, x, y):
    """Compute cross-entropy loss for language modeling."""
    # Set model weights
    for i, p in enumerate(params):
        model.trainable_weights[i].assign(p)
    
    # Forward pass
    logits = model(x, training=True)
    
    # Compute cross-entropy
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits,
        labels=y
    )
    return jnp.mean(loss)

# Training step (JIT compiled for speed)
@jax.jit
def train_step(params, opt_state, x, y):
    """Single training step."""
    loss, grads = jax.value_and_grad(compute_loss)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, params)  # Pass params here
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

print("Training setup complete")
print(f"Optimizer: AdamW with lr={CONFIG['learning_rate']}")
print(f"Training for {CONFIG['max_steps']} steps")

## Training Loop

Train the model by iterating through batches and updating parameters.

In [ ]:
# Training loop
losses = []
batch_size = CONFIG["batch_size"]
max_steps = CONFIG["max_steps"]

pbar = tqdm(range(max_steps), desc="Training")
for step in pbar:
    # Sample batch
    key, subkey = random.split(key)
    batch_x, batch_y = sample_batch(X, Y, batch_size, subkey)
    
    # Training step
    params, opt_state, loss = train_step(params, opt_state, batch_x, batch_y)
    
    # Track loss
    losses.append(float(loss))
    
    # Update progress bar
    pbar.set_postfix({"loss": f"{loss:.4f}"})
    
    # Print milestone updates
    if (step + 1) % 500 == 0:
        print(f"\nStep {step + 1}: loss = {loss:.4f}")

# Update model with final parameters
for i, p in enumerate(params):
    model.trainable_weights[i].assign(p)

print(f"\nTraining complete!")
print(f"Final loss: {losses[-1]:.4f}")

## Visualize Training Progress

Plot the loss curve to see how the model learned over time.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(losses, linewidth=1.5, color='#4ECDC4')
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Starting loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Improvement: {losses[0] - losses[-1]:.4f}")

## Text Generation

Now let's use our trained model to generate text! We'll implement **autoregressive sampling**: predict the next token, add it to the sequence, and repeat.

In [ ]:
# JIT compile the forward pass for fast inference
@jax.jit
def _forward_inference(context_array):
    """JIT-compiled forward pass for generation."""
    return model(context_array, training=False)

def generate_text(model, prompt, max_len=200, temperature=0.8):
    """Generate text from a prompt using the trained model."""
    # Encode prompt
    tokens = encode(prompt)
    seq_len = CONFIG["seq_len"]
    
    # Random key for sampling
    sample_key = random.PRNGKey(np.random.randint(0, 10000))
    
    # Pad initial context if needed (for consistent JIT shapes)
    if len(tokens) < seq_len:
        # Pad with zeros at the start
        padded = [0] * (seq_len - len(tokens)) + tokens
    else:
        padded = tokens[-seq_len:]
    
    # Generate tokens
    for _ in range(max_len):
        # Take last seq_len tokens
        context = padded[-seq_len:]
        context_array = jnp.array([context])
        
        # Get logits for next token (using JIT-compiled function)
        logits = _forward_inference(context_array)
        next_token_logits = logits[0, -1, :] / temperature
        
        # Sample from distribution (not greedy - allows diversity)
        sample_key, subkey = random.split(sample_key)
        next_token = random.categorical(subkey, next_token_logits)
        
        # Add to sequence
        tokens.append(int(next_token))
        padded.append(int(next_token))
    
    # Decode and return
    generated = decode(tokens[len(encode(prompt)):])
    return generated

# Test generation
prompt = "Frodo said"
generated = generate_text(model, prompt, max_len=200, temperature=0.8)

print(f"Prompt: '{prompt}'")
print(f"\nGenerated text:")
print(f"{prompt}{generated}")

## Generate Multiple Samples

Let's generate a few different samples to see the variety in the model's outputs.

In [ ]:
prompts = [
    "Frodo said",
    "The Ring",
    "Gandalf looked at",
    "The hobbits were",
]

for prompt in prompts:
    generated = generate_text(model, prompt, max_len=150, temperature=0.8)
    print(f"\n>>> {prompt}{generated[:180]}")
    print("-" * 80)

## Visualize Attention Patterns

Let's visualize what the attention mechanism learned by looking at attention weights for a sample sequence.

In [ ]:
# Get attention weights from first block
sample_text = "hello world this is a test"
sample_tokens = jnp.array([encode(sample_text[:CONFIG["seq_len"]])])

# Forward pass to get embeddings
tok_emb = model.token_embedding(sample_tokens)
pos = jnp.arange(sample_tokens.shape[1])
pos_emb = model.position_embedding(pos)
x = tok_emb + pos_emb

# Get attention from first block
first_block = model.blocks[0]
attn_layer = first_block.attn

# Manually compute attention weights
B, T, C = x.shape[0], x.shape[1], x.shape[2]
qkv = attn_layer.qkv_proj(x)
qkv = ops.reshape(qkv, (B, T, 3, attn_layer.num_heads, attn_layer.head_size))
qkv = ops.transpose(qkv, axes=[2, 0, 3, 1, 4])
q, k, v = qkv[0], qkv[1], qkv[2]

scores = ops.matmul(q, ops.transpose(k, axes=[0, 1, 3, 2]))
scores = scores / ops.sqrt(ops.cast(attn_layer.head_size, dtype=scores.dtype))

mask = ops.tril(ops.ones((T, T)))
mask = ops.cast(mask, dtype=scores.dtype)
scores = ops.where(mask == 0, -1e9, scores)
attn_weights = ops.softmax(scores, axis=-1)

# Plot attention for first head
attn_head_0 = np.array(attn_weights[0, 0])  # First sample, first head

plt.figure(figsize=(10, 8))
plt.imshow(attn_head_0, cmap='viridis')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position', fontsize=12)
plt.ylabel('Query Position', fontsize=12)
plt.title('Attention Pattern (Head 0, Block 0)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Brighter colors indicate stronger attention")
print("Note the causal (lower-triangular) structure - each position only attends to past")

## Key Takeaways

### What We Built

- A **GPT-2 style transformer** from scratch using Keras + JAX
- Character-level language model trained on Lord of the Rings text

### Key Concepts

1. **Embeddings**: Token embeddings (what) + Position embeddings (where) = input representation
2. **Self-Attention**: Each token attends to all previous tokens to build context
3. **Multi-Head Attention**: Multiple attention patterns learned in parallel
4. **Causal Masking**: Prevents looking at future tokens during training
5. **Residual Connections**: Help gradient flow in deep networks
6. **Layer Normalization**: Stabilizes training
7. **Autoregressive Generation**: Generate text one token at a time

### Why JAX?

- **JIT compilation**: Makes training much faster
- **Automatic differentiation**: Computes gradients automatically
- **Functional programming**: Clean and composable code

### Next Steps

- Try different architectures (more layers, larger embeddings)
- Implement temperature sampling for more diverse generation
- Add dropout for regularization
- Train on larger datasets
- Implement beam search or nucleus sampling